## Iniciando o Spark e importando bibliotecas

In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import to_date, lit, date_format

try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("AnaliseDadosCredit") \
    .master("spark://spark-master:7077") \
    .getOrCreate()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/13 15:13:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
credit_card_balance = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/data/raw/credit_card_balance.csv")
credit_card_balance.createOrReplaceTempView("credit_card_balance")

credit_card_balance.show(5, truncate=False)

+----------+----------+--------------+-----------+-----------------------+------------------------+--------------------+--------------------------+------------------------+-----------------------+-------------------+-------------------------+------------------------+-------------+--------------------+------------------------+--------------------+--------------------------+------------------------+-------------------------+--------------------+------+----------+
|SK_ID_PREV|SK_ID_CURR|MONTHS_BALANCE|AMT_BALANCE|AMT_CREDIT_LIMIT_ACTUAL|AMT_DRAWINGS_ATM_CURRENT|AMT_DRAWINGS_CURRENT|AMT_DRAWINGS_OTHER_CURRENT|AMT_DRAWINGS_POS_CURRENT|AMT_INST_MIN_REGULARITY|AMT_PAYMENT_CURRENT|AMT_PAYMENT_TOTAL_CURRENT|AMT_RECEIVABLE_PRINCIPAL|AMT_RECIVABLE|AMT_TOTAL_RECEIVABLE|CNT_DRAWINGS_ATM_CURRENT|CNT_DRAWINGS_CURRENT|CNT_DRAWINGS_OTHER_CURRENT|CNT_DRAWINGS_POS_CURRENT|CNT_INSTALMENT_MATURE_CUM|NAME_CONTRACT_STATUS|SK_DPD|SK_DPD_DEF|
+----------+----------+--------------+-----------+------------------

In [3]:
credit_card_balance.count()

3840312

In [4]:
credit_card_balance = spark.sql("""
                                Select
                                    SK_ID_PREV as PK_AGG_CREDIT,
                                    SK_ID_CURR,
                                    AMT_BALANCE as FVL_AMT_BALANCE_CREDIT,
                                    AMT_CREDIT_LIMIT_ACTUAL as FVL_AMT_CREDIT_LIMIT_ACTUAL_CREDIT,
                                    AMT_DRAWINGS_ATM_CURRENT as FVL_AMT_DRAWINGS_ATM_CURRENT_CREDIT,
                                    AMT_DRAWINGS_CURRENT as FVL_AMT_DRAWINGS_CURRENT_CREDIT,
                                    AMT_DRAWINGS_OTHER_CURRENT as FVL_AMT_DRAWINGS_OTHER_CURRENT_CREDIT,
                                    AMT_DRAWINGS_POS_CURRENT as FVL_AMT_DRAWINGS_POS_CURRENT_CREDIT,
                                    AMT_INST_MIN_REGULARITY as FVL_AMT_INST_MIN_REGULARITY_CREDIT,
                                    AMT_PAYMENT_CURRENT as FVL_AMT_PAYMENT_CURRENT_CREDIT,
                                    AMT_PAYMENT_TOTAL_CURRENT as FVL_AMT_PAYMENT_TOTAL_CURRENT_CREDIT,
                                    AMT_RECEIVABLE_PRINCIPAL as FVL_AMT_RECEIVABLE_PRINCIPAL_CREDIT,
                                    AMT_RECIVABLE as FVL_AMT_RECIVABLE_CREDIT,
                                    AMT_TOTAL_RECEIVABLE as FVL_AMT_TOTAL_RECEIVABLE_CREDIT,
                                    CNT_DRAWINGS_ATM_CURRENT as FVL_CNT_DRAWINGS_ATM_CURRENT_CREDIT,
                                    CNT_DRAWINGS_CURRENT as FVL_CNT_DRAWINGS_CURRENT_CREDIT,
                                    CNT_DRAWINGS_OTHER_CURRENT as FVL_CNT_DRAWINGS_OTHER_CURRENT_CREDIT,
                                    CNT_DRAWINGS_POS_CURRENT as FVL_CNT_DRAWINGS_POS_CURRENT_CREDIT,
                                    CNT_INSTALMENT_MATURE_CUM as FVL_CNT_INSTALMENT_MATURE_CUM_CREDIT,
                                    NAME_CONTRACT_STATUS as FC_NAME_CONTRACT_STATUS_CREDIT,
                                    SK_DPD as FVL_SK_DPD_CREDIT,
                                    SK_DPD_DEF as FVL_SK_DPD_DEF_CREDIT,
                                    cast(add_months('2023-12-01',MONTHS_BALANCE) as timestamp) as PK_DATREF_CREDIT,
                                    substr(translate(cast(add_months('2023-12-01',MONTHS_BALANCE) as string),'-',''),1,6) as PK_DATPART_CREDIT,
                                    MONTHS_BALANCE
                                from 
                                    credit_card_balance
                            """)
# Retirando valores nulos
credit_card_balance = credit_card_balance.where(col("MONTHS_BALANCE").isNotNull())

# Filtrando somente histórico necessário (15 meses)
stage01 = credit_card_balance.where(col("MONTHS_BALANCE") >= -200)
stage01 = stage01.drop("MONTHS_BALANCE")

stage01.createOrReplaceTempView("stage01")
stage01.count()

3840312

In [5]:
stage01.show(5, truncate=False)

+-------------+----------+----------------------+----------------------------------+-----------------------------------+-------------------------------+-------------------------------------+-----------------------------------+----------------------------------+------------------------------+------------------------------------+-----------------------------------+------------------------+-------------------------------+-----------------------------------+-------------------------------+-------------------------------------+-----------------------------------+------------------------------------+------------------------------+-----------------+---------------------+-------------------+-----------------+
|PK_AGG_CREDIT|SK_ID_CURR|FVL_AMT_BALANCE_CREDIT|FVL_AMT_CREDIT_LIMIT_ACTUAL_CREDIT|FVL_AMT_DRAWINGS_ATM_CURRENT_CREDIT|FVL_AMT_DRAWINGS_CURRENT_CREDIT|FVL_AMT_DRAWINGS_OTHER_CURRENT_CREDIT|FVL_AMT_DRAWINGS_POS_CURRENT_CREDIT|FVL_AMT_INST_MIN_REGULARITY_CREDIT|FVL_AMT_PAYMENT_CURRENT_CRED

In [6]:
spark.sql("""
            select
                PK_DATREF_CREDIT,
                PK_DATPART_CREDIT,
                count(*) as Volume
            from stage01
            group by 1,2
            order by  1 desc
""").show(200)

+-------------------+-----------------+------+
|   PK_DATREF_CREDIT|PK_DATPART_CREDIT|Volume|
+-------------------+-----------------+------+
|2023-11-01 00:00:00|           202311| 62356|
|2023-10-01 00:00:00|           202310| 94643|
|2023-09-01 00:00:00|           202309|100355|
|2023-08-01 00:00:00|           202308|102115|
|2023-07-01 00:00:00|           202307|100546|
|2023-06-01 00:00:00|           202306| 98577|
|2023-05-01 00:00:00|           202305| 95332|
|2023-04-01 00:00:00|           202304| 91419|
|2023-03-01 00:00:00|           202303| 86842|
|2023-02-01 00:00:00|           202302| 82525|
|2023-01-01 00:00:00|           202301| 78441|
|2022-12-01 00:00:00|           202212| 74705|
|2022-11-01 00:00:00|           202211| 71179|
|2022-10-01 00:00:00|           202210| 68458|
|2022-09-01 00:00:00|           202209| 65934|
|2022-08-01 00:00:00|           202208| 63466|
|2022-07-01 00:00:00|           202207| 61205|
|2022-06-01 00:00:00|           202206| 59071|
|2022-05-01 0

In [7]:
spark.sql("""
            select
                PK_DATREF_CREDIT,
                PK_DATPART_CREDIT,
                count(*) as Volume
            from stage01
            group by 1,2
            order by  1 desc
""").count()

96

#### Salvando tabela particionada (Parquet)

In [8]:
nm_path = '/data/processed/credit_card/'
stage01.write.partitionBy('PK_DATPART_CREDIT').parquet(nm_path, mode='overwrite')

In [9]:
spark.stop()